In [143]:
!pip install category_encoders lightgbm catboost xgboost

In [144]:
import pandas as pd
import numpy as np
import sklearn as sk
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
import category_encoders as ce
from collections import Counter
from scipy.special import logit
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

## 1. Download data from Don’tGetKicked competition. Design train/validation/test split.

In [145]:
!gdown 1ogHQ9uWJ-1Xl9F2jDoWAGV3Q-no6IVCq # train

Downloading...
From: https://drive.google.com/uc?id=1ogHQ9uWJ-1Xl9F2jDoWAGV3Q-no6IVCq
To: /content/training.csv
100% 14.5M/14.5M [00:00<00:00, 75.9MB/s]


In [146]:
train_data = pd.read_csv("training.csv")

In [147]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [148]:
train_data.isna().sum()

,0
RefId,0
IsBadBuy,0
PurchDate,0
Auction,0
VehYear,0
VehicleAge,0
Make,0
Model,0
Trim,2360
SubModel,8


In [149]:
# функция для очистки от NaN-ов
def nan_clear(data:pd.DataFrame):
  max_size = 0.5 * len(data)
  col_to_del = []
  for col in data.columns:
    if data[col].isna().sum() >= max_size:
      col_to_del.append(col)
  data = data.drop(col_to_del, axis=1)

  cat_cols = data.select_dtypes(include='object').columns.tolist()
  num_cols = data.select_dtypes(include=['int64',"float64"]).columns.tolist()

  for cat in cat_cols:
    if data[cat].isna().sum() != 0:
      data[cat] = data[cat].fillna(data[cat].mode()[0])

  for num in num_cols:
    if data[num].isna().sum() != 0:
      data[num] = data[num].fillna(data[num].mean())
  return data

In [150]:
train_data = nan_clear(train_data)
print(train_data.isna().sum().any()) # False - NaNов нет
train_data.shape

False


(72983, 32)

In [151]:
train_data.nunique()

,0
RefId,72983
IsBadBuy,2
PurchDate,517
Auction,3
VehYear,10
VehicleAge,10
Make,33
Model,1063
Trim,134
SubModel,863


- Биннинг для VehOdo (иначе долго будут считаться деревья)
- RefId удаляем, тк это ID машин.

In [152]:
train_data["VehOdo_Binned"] = pd.qcut(x=train_data["VehOdo"],
                                     q=[0, 0.25, 0.5, 0.75, 1.0],
                                     labels=[0, 1, 2 ,3])
train_data = train_data.drop(["VehOdo", "RefId"], axis=1)

- Use the "PurchDate" field to split, test must be later in time than validation, same goes for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate.
- Use the first 33% of the data for the training, the last 33% of the data for the test, and the middle 33% for the validation set. Don't use the test dataset until the end!

In [153]:
def tvt_split_by_time(train_df:pd.DataFrame):
  total = len(train_df)
  train_end_idx = int(total * 0.33)
  val_end_idx = int(total * 0.66)

  train_df["PurchDate"] = pd.to_datetime(train_df["PurchDate"], errors='coerce')
  train_df = train_df.sort_values(by="PurchDate")
  train_df["PurchDate"] = train_df["PurchDate"].astype(int)

  X = train_df.drop(["IsBadBuy"], axis=1)
  Y = train_df["IsBadBuy"]

  while X.loc[train_end_idx]["PurchDate"] == X.loc[val_end_idx]["PurchDate"]:
    val_end_idx +=1
    print(val_end_idx)

  X_train = X.iloc[:train_end_idx]
  X_valid = X.iloc[train_end_idx:val_end_idx]
  X_test = X.iloc[val_end_idx:]

  Y_train = Y.iloc[:train_end_idx]
  Y_valid = Y.iloc[train_end_idx:val_end_idx]
  Y_test = Y.iloc[val_end_idx:]
  return X_train, X_valid, X_test, Y_train, Y_valid, Y_test

In [154]:
X_train, X_valid, X_test, Y_train, Y_valid, Y_test = tvt_split_by_time(train_data)
len(X_train), len(X_valid), len(X_test)

(24084, 24084, 24815)

In [155]:
print(max(X_train["PurchDate"]))
print(max(X_valid["PurchDate"]))
print(max(X_test["PurchDate"]))

1252627200000000000
1273536000000000000
1293667200000000000


- Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables. Be careful with data leakage (fit Encoder to training and apply to validation & test).

In [156]:
def categories_transform(train_data:pd.DataFrame, X_train, X_valid, X_test):
  cat_cols = train_data.select_dtypes(include='object').columns.tolist()
  count_encoder = ce.CountEncoder(cols=cat_cols)
  train_encoded = count_encoder.fit_transform(X_train)
  valid_encoded = count_encoder.transform(X_valid)
  test_encoded = count_encoder.transform(X_test)
  return train_encoded, valid_encoded, test_encoded

In [157]:
train_encoded, valid_encoded, test_encoded = categories_transform(train_data, X_train, X_valid, X_test)
train_encoded.sample(5)

,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,WheelType,Nationality,Size,TopThreeAmericanName,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost,VehOdo_Binned
68713,1239235200000000000,4953,2006,3,4747,69,43,17,4162,23416,1.0,13326,20962,2947,8010,10767.0,12898.0,12128.0,14430.0,10767.0,12898.0,12128.0,14430.0,21973,32219,3373,9300.0,0,1118,1
57097,1233705600000000000,13960,2004,5,4225,30,783,4846,169,23416,2.0,10523,20962,2159,4588,2511.0,3512.0,3212.0,4293.0,3059.0,3835.0,3804.0,4642.0,16926,92057,2520,3365.0,0,920,0
8968,1233532800000000000,13960,2005,4,5612,626,244,244,2454,23416,1.0,13326,20962,9044,8364,3286.0,4307.0,4049.0,5152.0,4004.0,4842.0,4824.0,5729.0,20928,27542,2385,3400.0,0,1243,3
26323,1231286400000000000,13960,2006,3,388,283,5193,4846,1876,668,2.0,10523,2257,9044,3122,4333.0,5143.0,5180.0,6054.0,4492.0,5386.0,5351.0,6317.0,20928,32824,3373,4600.0,0,462,0
9168,1238976000000000000,13960,2005,4,2726,111,5193,4846,4162,23416,1.0,13326,20962,9044,8010,4012.0,5406.0,4833.0,6338.0,4012.0,5406.0,4833.0,6338.0,16044,27542,2385,5535.0,0,1020,1


In [158]:
train_encoded.nunique()

,0
PurchDate,180
Auction,3
VehYear,8
VehicleAge,8
Make,29
Model,128
Trim,69
SubModel,115
Color,16
Transmission,2


## 2. Create a Python class for Decision Tree Classifier and Decision Tree Regressor (MSE loss).
- It should support **fit**, **predict_proba**, and **predict** methods.
- Also, the maximum depth (max_depth) must be a parameter of your class.
- Use the Gini impurity criterion as a criterion for choosing the split.

- Create a separate class for Node. It should be able to hold data (sample features and targets), compute Gini impurity, and have pointers to children (left and right nodes).
- For the Regressor, use standard deviation instead of Gini impurity.
- Implement a function that finds the best possible split in the current node.
- Combine the previous steps into your working Decision Tree Classifier.
- Implement an Extra Randomized Tree by designing another function to find the best split.

In [159]:
class MyNode:
  def __init__(self, X, Y, feature=None, threshold=None, left=None, right=None,*,value=None, probs=None):
    self.X = X
    self.Y = Y
    self.feature = feature # разделением по какой фиче был получен
    self.threshold  = threshold # разделением по какому порогу был получен
    self.left = left # левый потомок
    self.right = right # правый потомок
    self.value = value # для листа
    self.class_probs = probs  # для листа

  def is_leaf_node(self):
    return self.value is not None

  def compute_gini(self):
    hist = np.bincount(self.Y)
    ps = hist / len(self.Y)
    return 1 - np.sum(ps ** 2)

  def compute_mse(self):
    n = len(self.Y)
    Y_mean = np.mean(self.Y)
    return np.sum((self.Y - Y_mean)**2)/n

In [160]:
class MyDecisionTreeClassifier:
  def __init__(self, min_samples_split=2, min_samples_leaf=1, max_depth=10, n_features=None, criterion='gini', random_state=21):
    self.min_samples_split = min_samples_split
    self.min_samples_leaf = min_samples_leaf
    self.max_depth = max_depth
    self.n_features = n_features
    self.criterion = criterion
    self.random_state = random_state
    self.root=None
    self.classes_=None

  def fit(self, X, Y):
    X = self._to_numpy(X)
    Y = self._to_numpy(Y)

    np.random.seed(self.random_state)
    self.classes_ = np.unique(Y)
    self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
    self.root = self._build_tree(X, Y)
    self._prune_tree(self.root)

  def predict(self, X):
    if self.root is None:
      raise ValueError("Дерево еще не обучено методом fit()!")
    X = self._to_numpy(X)
    return np.array([self._traverse_tree(x, self.root) for x in X])

  def predict_proba(self, X):
    if self.root is None:
      raise ValueError("Дерево еще не обучено методом fit()!")
    X = self._to_numpy(X)
    probas = [self._get_class_probs(x, self.root) for x in X]
    return np.array(probas)

  def _to_numpy(self, data):
    if isinstance(data, (pd.DataFrame, pd.Series)):
        return data.values
    elif isinstance(data, np.ndarray):
        return data
    else:
        return np.array(data)

  def _get_class_probs(self, x, node):
    if node.is_leaf_node():
        return node.class_probs

    if x[node.feature] <= node.threshold:
        return self._get_class_probs(x, node.left)
    return self._get_class_probs(x, node.right)

  def _traverse_tree(self, x, node):
    if node.is_leaf_node():
      return node.value

    if x[node.feature] <= node.threshold:
      return self._traverse_tree(x, node.left)
    return self._traverse_tree(x, node.right)

  def _build_tree(self, X, Y, depth=0):
    n_samples, n_feats = X.shape
    n_labels = len(np.unique(Y))

    # stopping criteria - base case
    if (depth >= self.max_depth or n_labels == 1
        or n_samples < self.min_samples_split
        or n_samples < 2 *self.min_samples_leaf):
      class_probs = self._compute_probs(Y)
      leaf_value = self._find_common_label(Y)
      return MyNode(X, Y, value=leaf_value, probs=class_probs)

    # recursive case
    # выбираем рандом фичи, replace=False чтобы избежать дублей
    feat_idx = np.random.choice(n_feats, self.n_features, replace=False)
    # find best split
    best_feature, best_threshold = self._find_best_split(X, Y, feat_idx)
    if best_feature is None:
      class_probs = self._compute_probs(Y)
      leaf_value = self._find_common_label(Y)
      return MyNode(X, Y, value=leaf_value, probs=class_probs)

    # create child nodes
    left_idxs, right_idxs = self._split(X[:, best_feature], best_threshold)
    left = self._build_tree(X[left_idxs, :], Y[left_idxs], depth+1)
    right = self._build_tree(X[right_idxs, :], Y[right_idxs], depth+1)
    return MyNode(X, Y, best_feature, best_threshold, left, right)

  def _find_common_label(self, Y):
    classes = Counter(Y)
    value = classes.most_common(1)[0][0] #[(obj, cnt)]
    return value

  def _compute_probs(self, Y):
    cnt = Counter(Y)
    probs = np.zeros(len(self.classes_))
    for i, cls in enumerate(self.classes_):
      probs[i] = cnt.get(cls, 0) / len(Y)
      # cnt.get(cls) returns None если класса в узле нет
    return probs


  def _find_best_split(self, X, Y, feat_idxs):
    split_idx, split_threshold = None, None

    if self.criterion == 'gain':
      # цель - уменьшение gain [0;+inf)
      best_value = -1
      compute_func = self._compute_gain

    elif self.criterion == 'gini':
      # цель - уменьшение gini [0;1]
      best_value = 0
      compute_func = self._compute_delta_gini
    else:
        return None, None

    for feat_idx in feat_idxs:
        X_column = X[:, feat_idx]

        thresholds = np.unique(X_column)

        # если уник значений много, выбираем как пороги только 100 по процентилям
        if len(thresholds) > 100:
          percentiles = np.linspace(0, 100, 100)
          thresholds = np.percentile(X_column, percentiles)

        for thr in thresholds:
            left_idxs, right_idxs = self._split(X_column, thr)
            # проверка на min_samples_leaf
            if len(left_idxs) < self.min_samples_leaf or len(right_idxs) < self.min_samples_leaf:
                continue

            current_value = compute_func(Y, X_column, thr)
            if current_value > best_value:
                best_value = current_value
                split_idx = feat_idx
                split_threshold = thr
    return split_idx, split_threshold

  def _split(self, X_column, split_threshold):
    # [[][][]] > []
    left_mask = X_column <= split_threshold
    right_mask = ~left_mask

    left_idxs = np.where(left_mask)[0]
    right_idxs = np.where(right_mask)[0]

    #left_idxs = np.argwhere(X_column<=split_threshold).flatten()
    #right_idxs =  np.argwhere(X_column>split_threshold).flatten()
    return left_idxs, right_idxs

  def _prune_tree(self, node):
        node.X = None
        if node.left:
            self._prune_tree(node.left)
        if node.right:
            self._prune_tree(node.right)

  def _compute_delta_gini(self, Y, X_column, threshold):
    left_mask = X_column <= threshold
    right_mask = ~left_mask

    n_left = np.sum(left_mask)
    n_right = np.sum(right_mask )

    if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
        return 0

    n = len(Y)
    # доля класса 1 (p) - для Y содерж только 0 и 1:
    p_parent = np.mean(Y)

    # дочерн узлы
    p_left = np.mean(Y[left_mask]) if n_left > 0 else 0
    p_right = np.mean(Y[right_mask]) if n_right > 0 else 0

    # gini для бин кл = 2 * p * (1 - p)
    gini_parent = 2 * p_parent * (1 - p_parent)
    gini_left = 2 * p_left * (1 - p_left) if n_left > 0 else 0
    gini_right = 2 * p_right * (1 - p_right) if n_right > 0 else 0
    delta_gini = gini_parent - (n_left/n) * gini_left - (n_right/n) * gini_right

    return delta_gini


  def _compute_gain(self, Y, X_column, threshold):
    # parent enthropy
    parent_enthropy = self._enthropy(Y)

    # create children
    left_idxs, right_idxs = self._split(X_column, threshold)
    if len(left_idxs) == 0 or len(right_idxs) == 0:
      return 0

    # calc weighted entr of children
    n = len(Y)
    n_left, n_right = len(left_idxs), len(right_idxs)
    ent_left, ent_right = self._enthropy(Y[left_idxs]), self._enthropy(Y[right_idxs])
    child_enthropy = (n_left/n) * ent_left + (n_right/n) * ent_right

    # IG
    inf_gain = parent_enthropy - child_enthropy
    return inf_gain

  def _enthropy(self, Y):
    hist = np.bincount(Y)
    ps = hist / len(Y)
    return -np.sum([p*np.log(p) for p in ps if p>0])



In [161]:
def gini_21(Y_true:np.array, Y_predicted:np.array):
  gini_score = 2 * roc_auc_score(Y_true, Y_predicted) - 1
  return gini_score

## 3. With your DecisionTree module, you must obtain a Gini score of at least 0.1 on the validation dataset.
- да, см. ниже
## 4. Use sklearn's DecisionTreeClassifier and check its performance on the validation dataset. Is it better than your module? If so, why?
- тоже см ниже

In [162]:
my_classifier = MyDecisionTreeClassifier(max_depth=15, min_samples_split=100, n_features=10)
my_classifier.fit(train_encoded, Y_train)

In [163]:
Y_pred_my_proba = my_classifier.predict_proba(valid_encoded)[:, 1]
my_gini = gini_21(Y_valid, Y_pred_my_proba)
print(f"My gini_score: {my_gini:.4f}")
#0.3533

My gini_score: 0.3533


In [164]:
skl_tree = DecisionTreeClassifier(max_depth=15, min_samples_split=100, max_features=10, random_state=21)
skl_tree.fit(train_encoded, Y_train)

DecisionTreeClassifier(max_depth=15, max_features=10, min_samples_split=100,
                       random_state=21)

In [165]:
Y_pred_skl_proba = skl_tree.predict_proba(valid_encoded)[:, 1]
skl_gini = gini_21(Y_valid, Y_pred_skl_proba)
print(f"Sklearn gini_score: {skl_gini:.4f}")
#0.3000

Sklearn gini_score: 0.3000


- Gini-score получается лучше, чем в реализации sklearn, потому что реализованный мной алгоритм использует случайный подвыбор признаков, а модель sklearn оптимизированный детерминированный набор порогов (по квантилям).

## Regressor

In [213]:
class MyDecisionTreeRegressor:
  def __init__(self, min_samples_split=2, min_samples_leaf=1, max_depth=10,n_features=None, criterion='squared_error', random_state=21):
    self.min_samples_split = min_samples_split
    self.min_samples_leaf = min_samples_leaf
    self.max_depth = max_depth
    self.n_features = n_features
    self.criterion = criterion
    self.random_state = random_state
    self.root=None
    self.initial_variance_=None

  def fit(self, X, Y):
    X = self._to_numpy(X)
    Y = self._to_numpy(Y)
    self.initial_variance_ = np.var(Y)
    np.random.seed(self.random_state)
    self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
    self.root = self._build_tree(X, Y)
    return self

  def predict(self, X):
    if self.root is None:
      raise ValueError("Дерево еще не обучено методом fit()!")
    X = self._to_numpy(X)
    return np.array([self._traverse_tree(x, self.root) for x in X])

  def _traverse_tree(self, x, node):
    if node.is_leaf_node():
      return node.value

    if x[node.feature] <= node.threshold:
      return self._traverse_tree(x, node.left)
    return self._traverse_tree(x, node.right)

  def _build_tree(self, X, Y, depth=0):
    n_samples, n_feats = X.shape
    min_variance_ratio = 0.01

    # base case
    if (depth >= self.max_depth
        or np.var(Y) < min_variance_ratio * self.initial_variance_
        or n_samples < self.min_samples_split
        or n_samples < 2 *self.min_samples_leaf):
      leaf_value = np.mean(Y)
      return MyNode(X, Y, value=leaf_value)

    # recursive case
    feat_idx = np.random.choice(n_feats, self.n_features, replace=False)
    best_feature, best_threshold = self._find_best_split(X, Y, feat_idx)
    if best_feature is None:
      leaf_value = np.mean(Y)
      return MyNode(X, Y, value=leaf_value)

    # create child nodes
    left_idxs, right_idxs = self._split(X[:, best_feature], best_threshold)
    left = self._build_tree(X[left_idxs, :], Y[left_idxs], depth+1)
    right = self._build_tree(X[right_idxs, :], Y[right_idxs], depth+1)
    return MyNode(X, Y, best_feature, best_threshold, left, right)

  def _find_best_split(self, X, Y, feat_idxs):
    split_idx, split_threshold = None, None
    if self.criterion == 'squared_error':
      # цель - уменьшение mse
      best_delta_mse = 0
      for feat_idx in feat_idxs:
        X_column = X[:, feat_idx]

        thresholds = np.unique(X_column)
        if len(thresholds) > 100:
          percentiles = np.linspace(0, 100, 100)
          thresholds = np.percentile(X_column, percentiles)

        for thr in thresholds:
          left_idxs, right_idxs = self._split(X_column, thr)

          if len(left_idxs) < self.min_samples_leaf or len(right_idxs) < self.min_samples_leaf:
              continue

          delta_mse = self._compute_delta_mse(Y, X_column, thr)
          if  delta_mse > best_delta_mse:
            best_delta_mse = delta_mse
            split_idx = feat_idx
            split_threshold = thr
    return split_idx, split_threshold

  def _split(self, X_column, split_threshold):
    left_mask = X_column <= split_threshold
    right_mask = ~left_mask

    left_idxs = np.where(left_mask)[0]
    right_idxs = np.where(right_mask)[0]
    return left_idxs, right_idxs

  def _compute_delta_mse(self,Y, X_column, threshold):
    left_mask = X_column <= threshold
    right_mask = ~left_mask

    n_left = np.sum(left_mask)
    n_right = np.sum(right_mask)
    n = len(Y)

    if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
        return 0.0

    parent_mse = np.var(Y, ddof=0)
    Y_left = Y[left_mask]
    Y_right = Y[right_mask]
    ss_left = np.sum((Y_left - np.mean(Y_left))**2) if n_left > 0 else 0.0
    ss_right = np.sum((Y_right - np.mean(Y_right))**2) if n_right > 0 else 0.0
    weighted_children_mse = (ss_left + ss_right) / n

    delta_mse = parent_mse - weighted_children_mse
    return max(delta_mse, 0.0)

  def _to_numpy(self, data):
    if isinstance(data, (pd.DataFrame, pd.Series)):
        return data.values
    elif isinstance(data, np.ndarray):
        return data
    else:
        return np.array(data)

In [167]:
my_regressor = MyDecisionTreeRegressor(max_depth=15, min_samples_split=100, n_features=10)
my_regressor.fit(train_encoded, Y_train)

- используем r2_score как некий аналог gini_score (показывает, насколько хорошо модель объясняет дисперсию целевой переменной)

In [168]:
Y_pred_my = my_regressor.predict(valid_encoded)
my_r2 = r2_score(Y_valid, Y_pred_my)
print(f"My r2_score on valid: {my_r2:.4f}")
# -0.0994

My r2_score on valid: -0.0994


In [169]:
Y_pred_my = my_regressor.predict(train_encoded)
my_r2 = r2_score(Y_train, Y_pred_my)
print(f"My r2_score on train: {my_r2:.4f}")
# 0.2951

My r2_score on train: 0.2951


- моя модель переобучается, и работает хуже обычного предсказывания среднего значения.

In [170]:
skl_rtree = DecisionTreeRegressor(max_depth=15, min_samples_split=100, max_features=10, random_state=21)
skl_rtree.fit(train_encoded, Y_train)

DecisionTreeRegressor(max_depth=15, max_features=10, min_samples_split=100,
                      random_state=21)

In [171]:
Y_pred_skl = skl_rtree.predict(valid_encoded)
skl_r2 = r2_score(Y_valid, Y_pred_skl)
print(f"Sklearn r2_score on valid: {skl_r2:.4f}")
# -0.3592

Sklearn r2_score on valid: -0.3592


In [172]:
Y_pred_skl = skl_rtree.predict(train_encoded)
skl_r2 = r2_score(Y_train, Y_pred_skl)
print(f"Sklearn r2_score on train: {skl_r2:.4f}")
# 0.2686

Sklearn r2_score on train: 0.2686


- Однако оригинал тоже переобучается, сильно подстраивается под тренировочный датасет и в итоге оказывается сильно хуже, чем просто предсказывать среднее значение.


- Получается, что мое дерево оказывается сильнее регуляризовано за счет дополнительного критерия остановки (останавливаемся, когда дисперсия в узле меньше 1% от начального значения).

## Extra-randomized Tree (Classifier)

- По сути, наследуем от класса обычного дерева с переопределением метода _find_best_split

In [173]:
class MyExtraRandomizedTree(MyDecisionTreeClassifier):
  def __init__(self, min_samples_split=2, min_samples_leaf=1, max_depth=10, n_features=None, criterion='gini', random_state=21, n_random_thresholds=5):
    super().__init__(min_samples_split=min_samples_split,
                         min_samples_leaf=min_samples_leaf,
                         max_depth=max_depth,
                         n_features=n_features,
                         criterion=criterion,
                         random_state=random_state)
    self.n_random_thresholds = n_random_thresholds

  def _find_best_split(self, X, Y, feat_idxs):
    split_idx, split_threshold = None, None

    if self.criterion == 'gain':
        best_value = -1
        compute_func = self._compute_gain
    elif self.criterion == 'gini':
        best_value = 0
        compute_func = self._compute_delta_gini
    else:
        return None, None

    for feat_idx in feat_idxs:
        X_column = X[:, feat_idx]
        unique_values = np.unique(X_column)
        n_possible = min(len(unique_values), self.n_random_thresholds)
        thresholds = np.random.choice(unique_values, size=n_possible, replace=False)
        for thr in thresholds:
            left_idxs, right_idxs = self._split(X_column, thr)

            if len(left_idxs) < self.min_samples_leaf or len(right_idxs) < self.min_samples_leaf:
                continue

            current_value = compute_func(Y, X_column, thr)

            if current_value > best_value:
                best_value = current_value
                split_idx = feat_idx
                split_threshold = thr

    return split_idx, split_threshold

In [174]:
my_classifier = MyExtraRandomizedTree(max_depth=15, min_samples_split=100, n_features=10)
my_classifier.fit(train_encoded, Y_train)

In [175]:
Y_pred_my_proba = my_classifier.predict_proba(valid_encoded)[:,1]
my_gini = gini_21(Y_valid, Y_pred_my_proba)
print(f"My gini_score: {my_gini:.4f}")
#0.3620

My gini_score: 0.3620


- чуточку лучше, чем в моей реализации обычного дерева (0.3533), но надо помнить, что оно было "полурандомным" (случайные признаки и перцентильные пороги)

## 5. Implement the RandomForestClassifier and check its performance.
- You have to improve the result of a single tree and get at least 0.15 Gini score on the validation dataset
- Be able to set a fixed random seed.

In [176]:
class MyRandomForestClassifier:
  def __init__(self, n_estimators=100, criterion='gini', max_depth=15,min_samples_split=100,min_samples_leaf=1,n_features=10, random_state=21, max_samples=None):
    self.n_estimators = n_estimators
    self.criterion = criterion
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.min_samples_leaf = min_samples_leaf
    self.n_features = n_features
    self.random_state = random_state
    self.max_samples = max_samples # абс число, не доля
    self.trees = []

  def fit(self, X, Y):
    X = self._to_numpy(X)
    Y = self._to_numpy(Y)
    self.classes_ = np.unique(Y)
    self.n_classes_ = len(self.classes_)
    self.trees = []
    rng = np.random.default_rng(self.random_state)

    for i in range(self.n_estimators):
      tree_seed = rng.integers(0, 2**31 - 1)
      X_sample, Y_sample = self._bootstrap_sample(X, Y, rng)
      tree = MyDecisionTreeClassifier(self.min_samples_split,
                                      self.min_samples_leaf,
                                      self.max_depth,
                                      self.n_features,
                                      self.criterion,
                                      random_state=tree_seed)
      tree.fit(X_sample, Y_sample)
      self.trees.append(tree)

  def predict(self, X):
    if not self.trees:
      raise ValueError("Лес еще не обучен методом fit()!")
    X = self._to_numpy(X)
    all_preds = np.array([tree.predict(X) for tree in self.trees])

    n_samples = X.shape[0]
    final_predictions = np.zeros(n_samples, dtype=self.classes_.dtype)

    for sample_idx in range(n_samples):
        # голоса всех деревьев для одного образца
        votes = all_preds[:, sample_idx]
        # голоса в индексы self.classes_
        vote_indices = np.searchsorted(self.classes_, votes)
        # голоса с учетом длины n_classes_
        counts = np.bincount(vote_indices, minlength=self.n_classes_)
        final_predictions[sample_idx] = self.classes_[np.argmax(counts)]
    return final_predictions

  def predict_proba(self, X):
    if not self.trees:
      raise ValueError("Лес еще не обучен методом fit()!")
    X = self._to_numpy(X)
    n_samples = X.shape[0]

    # вер-ти от ВСЕХ трис
    all_proba = np.zeros((self.n_estimators, n_samples, self.n_classes_))
    for i, tree in enumerate(self.trees):
        # от 1
        tree_proba = tree.predict_proba(X)  # (n_samples, n_classes)
        for j, cls in enumerate(tree.classes_):
            # индекс класса в общем списке классов леса
            class_index = np.where(self.classes_ == cls)[0]
            if len(class_index) > 0:
                all_proba[i, :, class_index[0]] = tree_proba[:, j]
    avg_proba = np.mean(all_proba, axis=0)
    return avg_proba

  def _bootstrap_sample(self, x, y, rng):
    n_samples = x.shape[0]
    sample_size = n_samples if not self.max_samples else min(self.max_samples, n_samples)
    idxs = rng.choice(n_samples, sample_size, replace=True)
    return x[idxs], y[idxs]

  def _to_numpy(self, data):
    if isinstance(data, (pd.DataFrame, pd.Series)):
        return data.values
    elif isinstance(data, np.ndarray):
        return data
    else:
        return np.array(data)

In [177]:
my_classifier = MyRandomForestClassifier(n_estimators=50,max_depth=15, min_samples_split=10, n_features=10, max_samples=100)
my_classifier.fit(train_encoded, Y_train)

In [178]:
Y_pred_my = my_classifier.predict_proba(valid_encoded)[:,1]
my_gini = gini_21(Y_valid, Y_pred_my)
print(f"My gini_score: {my_gini:.4f}")
# 0.4504 - если не ограничивать max_samples, но тогда считается 11 минут
# 0.1870

My gini_score: 0.1870


In [179]:
skl_forest = RandomForestClassifier(n_estimators=50,max_depth=15, min_samples_split=10, max_features=10, random_state=21, max_samples=100)
skl_forest.fit(train_encoded, Y_train)

RandomForestClassifier(max_depth=15, max_features=10, max_samples=100,
                       min_samples_split=10, n_estimators=50, random_state=21)

In [180]:
Y_pred_skl = skl_forest.predict_proba(valid_encoded)[:,1]
skl_gini = gini_21(Y_valid, Y_pred_skl)
print(f"Skl gini_score: {skl_gini:.4f}")
# 0.1834

Skl gini_score: 0.1834


- при равных параметрах моя реализация и sklearn сопоставимы.

## 6. Use your DecisionTree design class for GBDT classifier.
- This class must have max_depth, number_of_trees and max_features attributes.
- You must compute the gradient of the binary cross-entropy loss function and
- implement incremental learning: train the next tree using the results of the previous trees.

In [181]:
class MyGradientBoosting:
  def __init__(self, max_depth=15, number_of_trees=5, max_features=10, base_learner=MyDecisionTreeRegressor, random_state=21, learning_rate=0.001):
    self.max_depth = max_depth
    self.number_of_trees = number_of_trees
    self.max_features = max_features
    self.base_learner = base_learner
    self.random_state = random_state
    self.learning_rate = learning_rate

  def fit(self, X, Y):
    X = self._to_numpy(X)
    Y = self._to_numpy(Y)
    self.classes_ = np.unique(Y)
    self.n_classes_ = len(self.classes_)
    self.trees = []
    y = (Y == self.classes_[1]).astype(float)

    # нач предсказание
    pos = np.mean(y)
    F0 = np.full_like(y, logit(pos), dtype=float)

    # обуч деревья на ОСТАТКАХ
    self.X_train = X
    self.Y_train = y
    for i in range(self.number_of_trees):
      p = 1.0 / (1.0 + np.exp(-F0))
      res = y - p
      # min_samples_split=2, min_samples_leaf=1, max_depth=10,n_features=None, criterion='squared_error', random_state=21
      tree = self.base_learner(max_depth=self.max_depth, n_features= self.max_features, random_state = self.random_state+i)
      tree.fit(X, res)
      self.trees.append(tree)

      tree_pred = tree.predict(X)
      F0 += self.learning_rate * tree_pred
      p_curr = 1.0 / (1.0 + np.exp(-F0))
      curr_loss = self._compute_cross_enthropy(y, p_curr)
      print(f"Дерево {i}, Loss {curr_loss:.4f}")


  def predict(self, X):
    threshold = 0.5
    proba = self.predict_proba(X)
    result = (proba[:, 1] > threshold).astype(int)
    return result

  def predict_proba(self, X):
    X = self._to_numpy(X)
    pos = np.mean(self.Y_train)
    F = np.full(X.shape[0], logit(pos), dtype=float)

    for tree in self.trees:
      F += self.learning_rate * tree.predict(X)
    proba = 1.0 / (1.0 + np.exp(-F))
    result = np.column_stack([1-proba, proba])
    return result

  def _compute_cross_enthropy(self, y_true, pred):
    eps = 1e-15
    pred = np.clip(pred, eps, 1-eps)
    ce = - np.mean(y_true * np.log(pred) + (1 - y_true) * np.log(1 - pred))
    return ce

  def _to_numpy(self, data):
    if isinstance(data, (pd.DataFrame, pd.Series)):
        return data.values
    elif isinstance(data, np.ndarray):
        return data
    else:
        return np.array(data)

In [182]:
my_gbdt = MyGradientBoosting(max_depth=15, number_of_trees=5, max_features=10, base_learner=MyDecisionTreeRegressor, random_state=21, learning_rate=0.001)
my_gbdt.fit(train_encoded, Y_train)

Дерево 0, Loss 0.3556
Дерево 1, Loss 0.3556
Дерево 2, Loss 0.3555
Дерево 3, Loss 0.3554
Дерево 4, Loss 0.3554


In [183]:
Y_pred_my_proba = my_gbdt.predict_proba(valid_encoded)[:,1]
my_gini = gini_21(Y_valid, Y_pred_my_proba)
print(f"My gini_score: {my_gini:.4f}")
# 0.3587

My gini_score: 0.3587


In [184]:
skl_gbdt = GradientBoostingClassifier(max_depth=15, n_estimators=5, max_features=10, random_state=21, learning_rate=0.001,
                                      min_samples_split=2, min_samples_leaf=1)
skl_gbdt.fit(train_encoded, Y_train)

GradientBoostingClassifier(learning_rate=0.001, max_depth=15, max_features=10,
                           n_estimators=5, random_state=21)

In [185]:
Y_pred_skl_proba = skl_gbdt.predict_proba(valid_encoded)[:,1]
skl_gini = gini_21(Y_valid, Y_pred_skl_proba)
print(f"Skl gini_score: {skl_gini:.4f}")
# 0.3345

Skl gini_score: 0.3345


- Реализация медленная, но сопоставимая по качеству классификации при заданных параметрах.

## 7. Use LightGBM, Catboost, and XGBoost for fitting on a training set and prediction on a validation set.
- Review the documentation of the libraries and fine-tune the algorithms for the task.
- Note key differences between each implementation.
- Analyze special features of each algorithm (how does "categorical feature" work in Catboost, what is DART mode in XGBoost)?
- Which GBDT model gives the best result? Can you explain why?

# LightGBM
Особенности:
- Gradient-based One-Side Sampling (GOSS) - Односторонняя выборка на основе градиентов - для дальнейшего обучения сохраняет сэмплы данных с большей величиной ошибки и выборочно с малыми, что ускоряет обучение.
- Exclusive Feature Bundling (EFB) - Объединение Экслюзивных Признаков - объединяет разреженные взаимоисключающие признаки в один, что уменьшает размерность и опять-таки ускоряет обучение.

Плюсы:
- не нужен препроцессинг категориальных признаков
- высокая скорость
- масштабируемость расчетов (можно на GPU)

Минусы:
- на малых данных высок риск переобучения
- плохая интерпретируемость

### для LGBMClassifier
#### Отличительные параметры:
- boosting_type - какой тип бустинга используется (gbdt - градиентный, rf- рандом форест, dart - Dropouts meet Multiple Additive Regression Trees - когда случайная часть обученных деревьев убирается из последующих предсказаний - предотвращает переобучение)
 - subsample_for_bin - количество сэмплов для биннинга
 - objective - цель обучения и соответствующая функция потерь. (regression, binary, multiclass)
 - subsample, subsample_freq, colsample_bytree - доля строк и признаков для обучения каждого дерева
 - reg_alpha, reg_lambda - L1 и L2 регуляризации
 - importance_type - тип алгоритма для учета важности признаков (split - частота использования или gain - суммарный выигрыш)
 - linear_tree - меняет принцип работы алгоритма: вместо предсказания постоянного значения в листьях дерева (как в классическом GBDT), в каждом листе строится простая линейная регрессия на основе признаков, которые использовались при сплите до этого листа.


#### Обычные параметры:
- num_leaves
- max_depth
- learning_rate
- n_estimators
- class_weight (multiclass)
- min_split_gain  - мин уменьшение потерь для сплита
- min_child_weight, min_child_samples - мин кол-во данных в листе
- random_state
- n_jobs

In [186]:
train_df_for_lgb = pd.read_csv("training.csv")
train_df_for_lgb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [187]:
# заменяем nan на missing
cat_cols = train_df_for_lgb.select_dtypes(include=['object']).columns
date_cols = ['PurchDate']
cat_cols = [col for col in cat_cols if col not in date_cols]
train_df_for_lgb[cat_cols] = train_df_for_lgb[cat_cols].fillna('MISSING')

# указываем тип данных category
for column in cat_cols:
    train_df_for_lgb[column] = train_df_for_lgb[column].astype('category')

In [188]:
X_train_lgb, X_valid_lgb, X_test_lgb, Y_train_lgb, Y_valid_lgb, Y_test_lgb = tvt_split_by_time(train_df_for_lgb)

In [189]:
lgb_model = lgb.LGBMClassifier(max_depth=15, learning_rate=0.1, n_estimators=100, random_state=21, linear_tree=True)
lgb_model.fit(X_train_lgb, Y_train_lgb)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 2756, number of negative: 21328
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003307 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4487
[LightGBM] [Info] Number of data points in the train set: 24084, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.114433 -> initscore=-2.046240
[LightGBM] [Info] Start training from score -2.046240


LGBMClassifier(linear_tree=True, max_depth=15, random_state=21)

In [190]:
y_pred = lgb_model.predict_proba(X_valid_lgb)[:,1]
lgb_gini = gini_21(Y_valid_lgb, y_pred)
print(f"lgb gini_score: {lgb_gini:.4f}")
# 0.4342

lgb gini_score: 0.4342


# Catboost
Особенности:
- ordered boosting (упорядоченный бустинг) - обучение каждой последующей модели на ошибках предыдущей на "честном" градиенте - ошибке на части данных, которая предыдущая модель при обучении не видела. предотвращает переобучение.
- efficient handling of categorical features (работа с категориальными фичами) - не нужен препроцессинг в числовое представление.

Плюсы:
- не нужен препроцессинг категориальных признаков
- борьба с переобучением
- работает с NaN (но! сам обрабатывает в числовых, и требует явной замены строк на np.nan в категориальных)
- интерпретируемость (есть встроенные инструменты для feature importance, SHAP)

Минусы:
- сравнительно долгая длительность обучения
- повышенное потребление памяти

### Основные для CatboostClassifier
#### Отличительные параметры:
- cat_cols - индексы категориальных колонок
- l2_leaf_reg, random_strength, use_best_model, early_stopping_rounds - для регуляризации
- bootstrap_type - тип бутстрепа (Bayesian, Bernoulli, MVS, Poisson)
- subsample - доля объектов для обучения каждого дерева
- grow_policy - стратегия роста дерева (SymmetricTree (быстро, по умолчанию), Depthwise, Lossguide)
- one_hot_max_size - порог для one-hot кодирования категориальных признаков
- max_ctr_complexity - макс сложность комбинаций категор признаков для построения новых признаков
- task_type - GPU or CPU
- boosting_type - тип бустинга ('Ordered' (стандартный, защита от переобучения) или 'Plain' (классический градиентный бустинг))

#### Обычные параметры:
- iterations / n_estimators (взаимоисключающие)
- depth
- learning_rate
- loss_function
- max_leaves, min_data_in_leaf
- random_state


In [191]:
train_df_for_cb = pd.read_csv("training.csv")
train_df_for_cb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [192]:
# для чистоты эксперименты NaN не будем обрабатывать, только заменим на missing
cat_cols = train_df_for_cb.select_dtypes(include=['object']).columns
train_df_for_cb[cat_cols] = train_df_for_cb[cat_cols].fillna('MISSING')
X_train_cb, X_valid_cb, X_test_cb, Y_train_cb, Y_valid_cb, Y_test_cb = tvt_split_by_time(train_df_for_cb)

In [193]:
X_train_cb.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24084 entries, 32367 to 44671
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              24084 non-null  int64  
 1   PurchDate                          24084 non-null  int64  
 2   Auction                            24084 non-null  object 
 3   VehYear                            24084 non-null  int64  
 4   VehicleAge                         24084 non-null  int64  
 5   Make                               24084 non-null  object 
 6   Model                              24084 non-null  object 
 7   Trim                               24084 non-null  object 
 8   SubModel                           24084 non-null  object 
 9   Color                              24084 non-null  object 
 10  Transmission                       24084 non-null  object 
 11  WheelTypeID                        23175 non-null  floa

In [194]:
categorical_feature_indices = [i for i, cat in enumerate(X_train_cb.columns) if X_train_cb[cat].dtype == 'object']

In [195]:
catboost_model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0)
catboost_model.fit(X_train_cb, Y_train_cb, cat_features=categorical_feature_indices)

In [196]:
y_pred = catboost_model.predict_proba(X_valid_cb)[:,1]
catboost_gini = gini_21(Y_valid_cb, y_pred)
print(f"Catboost gini_score: {catboost_gini:.4f}")
# 0.4671

Catboost gini_score: 0.4671


# XGBoost = eXtreme Gradient Boosting
Особенности:
- с версии 1.5 не нужен препроцессинг категориальных признаков
- несколько встроенных техник для предотвращения переобучения: темп обучения, регуляризация, "подрезка" деревьев.
- построение деревьев: не в глубину, а в ширину
- работает с NaN
- кэширование
- использует приближененные вычисления, а не "жадный" алгоритм при вычислении наиболее благоприятного разделения признаков

Плюсы:
- хорош для больших датасетов
- поддержка паралеллизма и расчетов на GPU
- кастомизируемые параметры и регуляризация
- есть встроенные инструменты для feature importance


Минусы:
 - вычислительные затраты
 - чувствителен к шуму и выбросам
 - ограниченная интепретируемость

### Основные для XGBClassifier
#### Отличительные параметры:
- enable_categorical - поддержка категор признаков
- booster - тип модели (gbtree (по умолчанию), dart (с дропаутом), gblinear)
- tree_method -  алгоритм построения дерева (auto, exact, approx, hist, gpu_hist)
- importance_type - алгоритм расчета важности признаков (weight (частота), gain, cover, total_gain, total_cover)

- subsample, colsample_bytree, colsample_bylevel, colsaple_bynode -  доли объектов и признаков для построения каждого дерева, доли признаков для каждого уровня и узла
- scale_pos_weight -  вес для положителього класса (важен при дисбалансе)
- reg_lambda, reg_alpha - L2, L1 регуляризации
- gamma (min split loss) - мин уменьшение потерь для сплита
- device - GPU or CPU



#### Обычные параметры:
- n_estimators
- max_depth
- learning_rate (eta)
- objective
- random_state

In [197]:
train_df_for_xgb = pd.read_csv("training.csv")
train_df_for_xgb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [198]:
# заменяем nan на missing
cat_cols = train_df_for_xgb.select_dtypes(include=['object']).columns
date_cols = ['PurchDate']
cat_cols = [col for col in cat_cols if col not in date_cols]
train_df_for_xgb[cat_cols] = train_df_for_xgb[cat_cols].fillna('MISSING')

# указываем тип данных category
for column in cat_cols:
    train_df_for_xgb[column] = train_df_for_xgb[column].astype('category')

In [199]:
X_train_xgb, X_valid_xgb, X_test_xgb, Y_train_xgb, Y_valid_xgb, Y_test_xgb = tvt_split_by_time(train_df_for_xgb)

In [200]:
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    enable_categorical=True,
    n_estimators=50,
    max_depth=6,
    learning_rate=0.1,
    random_state=21
)
xgb_model.fit(X_train_xgb, Y_train_xgb)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=50,
              n_jobs=None, num_parallel_tree=None, ...)

In [201]:
y_pred = xgb_model.predict_proba(X_valid_xgb)[:,1]
xgb_gini = gini_21(Y_valid_cb, y_pred)
print(f"Xgb gini_score: {xgb_gini:.4f}")
# 0.4513

Xgb gini_score: 0.4513


- В целом, все три модели сопоставимы, но немного лучший результат показал CatBoost - как эталон для работы с датасетами, в которых много категориальных признаков.

## 8. Take the best model and estimate its performance on the test dataset:
- check the Gini values on all three datasets for your best model: training Gini, valid Gini, test Gini.
- Do you see a drop in performance when comparing the valid quality to the test quality?
- Is your model overfitting or not? Explain.

In [202]:
y_pred = catboost_model.predict_proba(X_train_cb)[:,1]
catboost_gini = gini_21(Y_train_cb, y_pred)
print(f"Catboost train valid gini_score: {catboost_gini:.4f}")

Catboost train valid gini_score: 0.7397


In [203]:
y_pred = catboost_model.predict_proba(X_valid_cb)[:,1]
catboost_gini = gini_21(Y_valid_cb, y_pred)
print(f"Catboost valid gini_score: {catboost_gini:.4f}")

Catboost valid gini_score: 0.4671


In [204]:
y_pred = catboost_model.predict_proba(X_test_cb)[:,1]
catboost_gini = gini_21(Y_test_cb, y_pred)
print(f"Catboost test gini_score: {catboost_gini:.4f}")

Catboost test gini_score: 0.4371


- Переобучение есть, тк качество метрики на валидационном и тестовом сете ниже примерно в 1.5 раза. Можно попробовать улучшить ситауцию с помощью встроенных механизмов:

In [205]:
catboost_model = CatBoostClassifier(
    iterations=1000,
    use_best_model=True,
    od_type='Iter',
    od_wait=20,
    eval_metric='AUC',
    learning_rate=0.1,
    depth=6,
    verbose=0)
catboost_model.fit(X_train_cb, Y_train_cb,
                   cat_features=categorical_feature_indices,
                   eval_set=(X_valid_cb, Y_valid_cb),
                   verbose=50)

0:	test: 0.6929583	best: 0.6929583 (0)	total: 124ms	remaining: 2m 3s
50:	test: 0.7431260	best: 0.7431271 (49)	total: 4.7s	remaining: 1m 27s
Stopped by overfitting detector  (20 iterations wait)

bestTest = 0.7433531497
bestIteration = 73

Shrink model to first 74 iterations.


In [206]:
y_pred = catboost_model.predict_proba(X_train_cb)[:,1]
catboost_gini = gini_21(Y_train_cb, y_pred)
print(f"Catboost train valid gini_score: {catboost_gini:.4f}")

Catboost train valid gini_score: 0.5682


In [207]:
y_pred = catboost_model.predict_proba(X_valid_cb)[:,1]
catboost_gini = gini_21(Y_valid_cb, y_pred)
print(f"Catboost valid gini_score: {catboost_gini:.4f}")

Catboost valid gini_score: 0.4867


In [208]:
y_pred = catboost_model.predict_proba(X_test_cb)[:,1]
catboost_gini = gini_21(Y_test_cb, y_pred)
print(f"Catboost test gini_score: {catboost_gini:.4f}")

Catboost test gini_score: 0.4823


- Вот теперь переобучения практически нет (валидационные и тестовые метрики близки, разрыв между метриками на тренировочном и валидационном датасете уменьшился)

## 9*. Implement the ExtraTreesClassifier and check its performance. You must improve the result of a single tree and obtain a Gini score of at least 0.12 on the validation dataset.

- ? наследование от RandomForest

In [209]:
class MyExtraTreesClassifier(MyRandomForestClassifier):
    def __init__(self, n_estimators=100, criterion='gini', max_depth=15,
                 min_samples_split=100, min_samples_leaf=1, n_features=10,
                 random_state=21, max_samples=None, n_random_thresholds=5):
        super().__init__(n_estimators=n_estimators,
                        criterion=criterion,
                        max_depth=max_depth,
                        min_samples_split=min_samples_split,
                        min_samples_leaf=min_samples_leaf,
                        n_features=n_features,
                        random_state=random_state,
                        max_samples=max_samples)
        self.n_random_thresholds = n_random_thresholds

    def fit(self, X, Y):
        X = self._to_numpy(X)
        Y = self._to_numpy(Y)
        self.classes_ = np.unique(Y)
        self.n_classes_ = len(self.classes_)
        self.trees = []
        rng = np.random.default_rng(self.random_state)

        for i in range(self.n_estimators):
            tree_seed = rng.integers(0, 2**31 - 1)
            X_sample, Y_sample = self._bootstrap_sample(X, Y, rng)

            tree = MyExtraRandomizedTree(
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_depth=self.max_depth,
                n_features=self.n_features,
                criterion=self.criterion,
                random_state=tree_seed,
                n_random_thresholds=self.n_random_thresholds
            )

            tree.fit(X_sample, Y_sample)
            self.trees.append(tree)

In [210]:
my_classifier = MyExtraTreesClassifier(n_estimators=50,max_depth=15, min_samples_split=10, n_features=10, max_samples=100, n_random_thresholds=5)
my_classifier.fit(train_encoded, Y_train)

In [211]:
Y_pred_my = my_classifier.predict_proba(valid_encoded)[:,1]
my_gini = gini_21(Y_valid, Y_pred_my)
print(f"My gini_score: {my_gini:.4f}")
# 0.23 - 0.25

My gini_score: 0.2459


In [212]:
skl_classifier = ExtraTreesClassifier(n_estimators=50,max_depth=15, min_samples_split=10, max_features=10, bootstrap=True, max_samples=100)
skl_classifier.fit(train_encoded, Y_train)

Y_pred_skl = skl_classifier.predict_proba(valid_encoded)[:,1]
skl_gini = gini_21(Y_valid, Y_pred_skl)
print(f"Skl gini_score: {skl_gini:.4f}")
# 0.31 - 0.39

Skl gini_score: 0.3108
